# 21 — EDA 보충 (발표 자료 p2 · p3 숫자 근거)

발표 준비 중 계산한 EDA 숫자를 모두 이 노트북에서 재현한다. 원본 CSV만 사용하고 다른 노트북에 의존하지 않는다.

| 절 | 발표 자료 숫자 |
|---|---|
| 1. view_5d 결측 | 6,887개(17.9%) 결측 · 07-01 ~ 07-09 100% · 8월 약 3% |
| 2. 8월 말 급등 | 08-26 이후 신규 롱폼 약 2.2배 · 좋아요·댓글 불변 · 쇼츠 변화 없음 · 측정 시점 동일 |
| 3. 평균회귀 | 과거 중앙값 5구간별 Rising 비율 (발표 p3: 31.3 / 25.9 / 15.9 / 17.7 / 10.1%) |

In [1]:
import os, glob
import duckdb
import numpy as np
import pandas as pd

RAW = os.environ.get('EPOCH_RAW', '../data/raw/epoch_data')
pd.set_option('display.width', 200)
con = duckdb.connect()
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    con.sql(f"CREATE OR REPLACE VIEW {os.path.splitext(os.path.basename(p))[0]} AS "
            f"SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")

## 1. view_5d 결측

`videos.csv`(전체 영상 38,411개) 중 `video_5d_views.csv`에 조회수가 없는 영상을 결측으로 본다.

In [2]:
miss = con.sql("""
    SELECT v.video_id, v.published_at, v.is_shorts, v.is_available, f.view_5d IS NULL AS missing
    FROM videos v LEFT JOIN video_5d_views f USING (video_id)""").df()
print(f'전체 영상 {len(miss):,}개 중 view_5d 결측 {miss.missing.sum():,}개 ({miss.missing.mean():.1%})')

miss['구간'] = np.select([miss.published_at < '2026-07-10', miss.published_at < '2026-08-01'],
                         ['07-01 ~ 07-09', '07-10 ~ 07-31'], '08-01 ~ 08-31')
print(miss.groupby('구간').missing.agg(영상수='size', 결측수='sum', 결측률='mean').round(3))

전체 영상 38,411개 중 view_5d 결측 6,887개 (17.9%)
                 영상수   결측수    결측률
구간                               
07-01 ~ 07-09   2096  2096  1.000
07-10 ~ 07-31  14051  4150  0.295
08-01 ~ 08-31  22264   641  0.029


In [3]:
# 업로드일별 결측률 (7월 초에 몰려 있는지, 8월 말에도 결측이 많은지)
d = miss.assign(업로드일=miss.published_at.dt.date).groupby('업로드일').missing.agg(영상수='size', 결측률='mean').round(2)
print(d.head(16).to_string()); print('...'); print(d.tail(8).to_string())

            영상수   결측률
업로드일                 
2026-07-01  163  1.00
2026-07-02  215  1.00
2026-07-03  237  1.00
2026-07-04  268  1.00
2026-07-05  237  1.00
2026-07-06  220  1.00
2026-07-07  235  1.00
2026-07-08  252  1.00
2026-07-09  269  1.00
2026-07-10  265  0.39
2026-07-11  286  0.00
2026-07-12  313  0.00
2026-07-13  577  0.57
2026-07-14  721  0.62
2026-07-15  756  0.64
2026-07-16  748  0.60
...
            영상수   결측률
업로드일                 
2026-08-24  676  0.02
2026-08-25  700  0.03
2026-08-26  767  0.04
2026-08-27  767  0.03
2026-08-28  859  0.03
2026-08-29  745  0.03
2026-08-30  779  0.03
2026-08-31  660  0.02


## 2. 8월 말 급등

**채널 평소 수준** = 07-26 ~ 08-19 업로드 롱폼의 log(1 + view_5d) 중앙값 (10_deflation.ipynb 1절과 같은 정의).
영상마다 `log(1 + view_5d) − 채널 평소 수준`(상승폭)을 구하고, 업로드일별 중앙값을 본다. 배수 = e^(상승폭).

**"약 2.2배"의 정의**: 08-27 ~ 08-31 업로드 롱폼 전체의 상승폭 중앙값 → e^0.797 = **2.22배**. 08-26은 상승이 시작되는 날(1.49배)이라 따로 표시한다.

In [4]:
BASE = """c AS (SELECT channel_id, is_shorts, MEDIAN(LN(1 + view_5d)) AS cm,
                    MEDIAN(LN(1 + like_5d) - LN(1 + view_5d)) AS clr
             FROM video_5d_views WHERE view_5d IS NOT NULL
               AND published_at >= TIMESTAMP '2026-07-26' AND published_at < TIMESTAMP '2026-08-20' GROUP BY 1, 2)"""
x = con.sql(f"""WITH {BASE}
    SELECT f.*, LN(1 + f.view_5d) - c.cm AS rel, (LN(1 + f.like_5d) - LN(1 + f.view_5d)) - c.clr AS like_rate_chg
    FROM video_5d_views f JOIN c USING (channel_id, is_shorts)
    WHERE f.view_5d IS NOT NULL AND f.published_at >= TIMESTAMP '2026-08-15'""").df()
x['업로드일'] = x.published_at.dt.date
L, Sh = x[x.is_shorts == 0], x[x.is_shorts == 1]

ch_up = L.groupby(['업로드일', 'channel_id']).rel.median().gt(0.3).groupby(level=0).mean()   # 그날 영상이 평소보다 뚜렷이 높은 채널 비율
daily = pd.DataFrame({'롱폼 수': L.groupby('업로드일').size(),
                      '롱폼 상승폭(log)': L.groupby('업로드일').rel.median(),
                      '롱폼 배수': np.exp(L.groupby('업로드일').rel.median()),
                      '쇼츠 상승폭(log)': Sh.groupby('업로드일').rel.median(),
                      '좋아요율 변화(log)': L.groupby('업로드일').like_rate_chg.median(),
                      '측정 시점 h_before': L.groupby('업로드일').h_before.median(),
                      '측정 시점 h_after': L.groupby('업로드일').h_after.median(),
                      '상승 채널 비율': ch_up})
print(daily.round(2).to_string())

for name, lo in [('08-26 하루', ('2026-08-26', '2026-08-27')), ('08-27 ~ 08-31', ('2026-08-27', '2026-09-01'))]:
    r = L[(L.published_at >= lo[0]) & (L.published_at < lo[1])].rel
    print(f'{name}: 롱폼 {len(r):,}개, 상승폭 중앙값 {r.median():.3f} → {np.exp(r.median()):.2f}배')

            롱폼 수  롱폼 상승폭(log)  롱폼 배수  쇼츠 상승폭(log)  좋아요율 변화(log)  측정 시점 h_before  측정 시점 h_after  상승 채널 비율
업로드일                                                                                                    
2026-08-15   513        -0.00   1.00        -0.03          0.00          119.95         120.10      0.20
2026-08-16   438         0.01   1.01        -0.03         -0.02          119.96         120.08      0.26
2026-08-17   418         0.01   1.01        -0.02          0.00          119.96         120.09      0.26
2026-08-18   439         0.00   1.00         0.00          0.00          119.89         120.04      0.25
2026-08-19   481        -0.01   0.99        -0.12         -0.01          119.87         120.04      0.21
2026-08-20   483        -0.05   0.95         0.08         -0.03          119.96         120.04      0.27
2026-08-21   488         0.02   1.02        -0.06         -0.02          119.84         120.03      0.33
2026-08-22   497        -0.02   0.99         0.03      

In [5]:
# 좋아요·댓글은 늘었나? — 08-27 ~ 08-31 롱폼, 채널 평소(07-26 ~ 08-19 롱폼 중앙값)와 원 단위 비교, 채널 규모 4분위별
cmp = con.sql("""
    WITH lv AS (SELECT * FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL),
    c AS (SELECT channel_id, MEDIAN(view_5d) AS mv, MEDIAN(like_5d) AS ml, MEDIAN(comment_5d) AS mc FROM lv
          WHERE published_at >= TIMESTAMP '2026-07-26' AND published_at < TIMESTAMP '2026-08-20' GROUP BY 1),
    x AS (SELECT lv.*, c.*, NTILE(4) OVER (ORDER BY c.mv) AS q FROM lv JOIN c USING (channel_id)
          WHERE published_at >= TIMESTAMP '2026-08-27')
    SELECT q AS 채널규모, COUNT(*) AS 영상수,
           MEDIAN(mv) AS 평소조회, MEDIAN(view_5d) AS 급등후조회,
           MEDIAN(ml) AS 평소좋아요, MEDIAN(like_5d) AS 급등후좋아요,
           MEDIAN(mc) AS 평소댓글, MEDIAN(comment_5d) AS 급등후댓글,
           MEDIAN(LN(1 + view_5d) - LN(1 + mv)) AS 상승폭_log
    FROM x GROUP BY 1 ORDER BY 1""").df()
cmp['채널규모'] = cmp.채널규모.map({1: '소형', 2: '중소형', 3: '중대형', 4: '대형'})
print(cmp.round(2).to_string(index=False))

채널규모  영상수     평소조회    급등후조회  평소좋아요  급등후좋아요  평소댓글  급등후댓글  상승폭_log
  소형  643    92.00    376.0    2.0     2.0  0.00    0.0     1.33
 중소형  643  3232.00   7492.0   55.0    43.5  3.00    4.0     0.95
 중대형  643 14223.00  30864.0  178.0   178.5 26.50   27.0     0.78
  대형  642 65301.75 118864.5  909.0   764.0 96.75   86.5     0.52


## 3. 평균회귀 — 과거 중앙값 5구간별 Rising 비율

**(a) 02_rule_baseline.ipynb 셀 10(그림만 있고 숫자 출력 없음)의 정의를 그대로 재현.**
발표 p3 숫자(31.3 / 25.9 / 15.9 / 17.7 / 10.1)와 구간마다 0.1 ~ 0.2%p 차이가 난다 — 02의 저장된 코드로는 발표 숫자를 정확히 재현할 수 없으므로(그림 작성 시 표본 수가 약간 달랐던 것으로 보임), **발표 자료는 이 재현값으로 바꾸는 것을 권장**한다. 추세·결론은 같다.
- 08-11 공식 라벨 601채널, `past_med` = 기준일 121시간 이전에 업로드된 롱폼의 view_5d 중앙값 (02 노트북 `CUT_HOURS = 121`)
- `pd.qcut(rank(method='first'), 5)`로 5구간

**(b) 참고**: 공식 라벨 정의(기준일 이전 롱폼 전체)로 08-11과 08-25를 계산. 추세는 같고, 08-25(급등기)에서 차이가 훨씬 크다.

In [6]:
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
tl['ref_date'] = tl.row_id.str.rsplit('_', n=1).str[1]
LABELS = ['하위20%', '20-40', '40-60', '60-80', '상위20%']

def quintile_rate(ref, cut_hours):
    pm = con.sql(f"""SELECT channel_id, MEDIAN(view_5d) AS past_med FROM video_5d_views
                     WHERE is_shorts = 0 AND view_5d IS NOT NULL
                       AND published_at < TIMESTAMP '{ref}' - INTERVAL {cut_hours} HOUR GROUP BY 1""").df()
    e = tl[tl.ref_date == ref].merge(pm, on='channel_id', how='left').dropna(subset=['past_med'])
    b = pd.qcut(e.past_med.rank(method='first'), 5, labels=LABELS)
    return (e.groupby(b, observed=True).target.mean() * 100).round(1), len(e)

r02, n02 = quintile_rate('2026-08-11', 121)
r11, n11 = quintile_rate('2026-08-11', 0)
r25, n25 = quintile_rate('2026-08-25', 0)
print(pd.DataFrame({f'(a) 02 정의 · 08-11 ({n02}채널) — 발표 p3': r02,
                    f'(b) 공식 정의 · 08-11 ({n11}채널)': r11,
                    f'(b) 공식 정의 · 08-25 ({n25}채널)': r25}).T.to_string())

past_med                           하위20%  20-40  40-60  60-80  상위20%
(a) 02 정의 · 08-11 (601채널) — 발표 p3   31.4   25.8   15.8   17.5   10.0
(b) 공식 정의 · 08-11 (601채널)           30.6   28.3   16.7   14.2   10.8
(b) 공식 정의 · 08-25 (666채널)           45.5   29.3   14.3    9.0    2.3


## 4. 정리

- **결측**: 영상 38,411개 중 6,887개(17.9%) view_5d 결측. 07-01 ~ 07-09 업로드는 100%(수집 시작 전), 8월은 약 3%. "5일 미경과" 때문이 아님 (08-26 ~ 31 업로드도 결측 약 3%).
- **급등**: 08-27 ~ 31 신규 롱폼 조회수가 채널 평소의 **약 2.2배**(일별 2.1 ~ 2.3배), 08-26은 1.5배로 상승 시작. 쇼츠는 변화 없음, 좋아요·댓글 수는 그대로(→ 좋아요율 하락), 측정 시점은 업로드 후 약 120시간으로 동일, 상승 채널 비율 20 ~ 30% → 70 ~ 78%.
- **평균회귀**: 과거 중앙값 하위 20% Rising 31.4% vs 상위 20% 10.0% (08-11, 02 정의 재현값 — 발표 p3의 31.3 / 10.1과 0.1%p 차이). 08-25(급등기, 공식 정의)에서는 45.5% vs 2.3%로 더 강함.